In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import datetime
from tqdm import tqdm
import time
import ast


pd.set_option('display.max_columns', 500)
# disable warnings in ipython
import warnings

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Sampler, DataLoader
from collections import defaultdict
import random

from torch.nn.utils.rnn import pad_sequence

# aether
import rasterio


In [ ]:
city = 'london'
loc_embedding_type = 'aether'  # {'calliper', 'aether', ...}

# create a separate directory for each city and embedding type if it doesn't exist
save_dir = f'pretrained_ace/{city}_{loc_embedding_type}'
os.makedirs(save_dir, exist_ok=True)


# Load the pre-trained location embedding model


## The AETHER model


In [ ]:
# Read the pre-trained AETHER model -- stored as a GeoTIFF file
# aether_london_tif_path = '/home/jliu/code/AE/data/AE/London_Embedding_AETHER_t3l.tif'
aether_london_tif_path = '/home/xlwang/code/2026/AETHER/outputs/tri2_AEProj-TextProj_LD_t3l_pix4-aug9_tau0.07-0.07_lam0.20_h256_l1_d128_bs512_lr1e-03_epo100/AETHER_embedding.tif'

with rasterio.open(aether_london_tif_path) as src:
    # Read the data into a numpy array
    data = src.read()  # Read all bands
    print(data.shape)  # Print the shape of the data


# Data Preparation


In [ ]:
stay_data_path = 'mobility_data/london_20250210_20250223_stays_bbox_min2staysperday_aether.csv.gz'


In [ ]:
df_stay_filtered = pd.read_csv(stay_data_path)
if "date" not in df_stay_filtered.columns:
    df_stay_filtered["date"] = pd.to_datetime(df_stay_filtered["start_time"]).dt.date.astype(str)
    
# Sort to ensure chronological order within a trajectory
df_stay_filtered = df_stay_filtered.sort_values(by=['user_id', 'date', 'start_time'])

# For London, we try projected CRS
gdf_stay_filtered = gpd.GeoDataFrame(df_stay_filtered, 
                                     geometry=gpd.points_from_xy(df_stay_filtered['longitude'], 
                                                                 df_stay_filtered['latitude'], 
                                                                 crs=4326))
gdf_stay_filtered = gdf_stay_filtered.to_crs("EPSG:27700")
gdf_stay_filtered['x'] = gdf_stay_filtered.geometry.x
gdf_stay_filtered['y'] = gdf_stay_filtered.geometry.y

df_stay_filtered = gdf_stay_filtered.drop(columns=['geometry'])

del gdf_stay_filtered
df_stay_filtered


In [ ]:
# if loc_embedding_type == 'calliper':

#     # coords = torch.tensor(df_stay_filtered[['longitude', 'latitude']].values, dtype=torch.float64).to(device)
#     coords = torch.tensor(df_stay_filtered[['x', 'y']].values, dtype=torch.float64).to(device)

#     # Get embeddings
#     # We get embeddings by calling the pretrained location encoder on the coordinates
#     # because saving the embeddings directly would take too much space.
#     # Moreover, it takes a long time (> 40 sec) to read the saved .csv.gz file 
#     # and a longer time to .apply(ast.literal_eval) (> 5 min)
#     # Whereas getting embeddings via the model is much faster (~20 sec on wye)
#     with torch.no_grad():
#         embeddings = pretrained_loc_encoder(coords).cpu().numpy()

#     # Add the embeddings to the dataframe as a new column named 'calliper_embedding'
#     embeddings_list = embeddings.tolist()
#     df_stay_filtered['calliper_embedding'] = embeddings_list

# elif loc_embedding_type == 'aether':
print("Using AETHER embeddings from the GeoTIFF file...")
# Use the columns 'row' and 'col' to index into the AETHER embedding array to get the corresponding embedding for each stay
# Note: the AETHER embedding array has shape (num_bands, height, width), so we need to index it as data[:, row, col]
# We will store the AETHER embeddings in a new column named 'aether_embedding'
aether_embeddings = []
for idx, row in df_stay_filtered.iterrows():
    row_idx = int(row['row'])
    col_idx = int(row['col'])
    embedding = data[:, row_idx, col_idx]  # Get the embedding for this stay
    aether_embeddings.append(embedding)
df_stay_filtered['aether_embedding'] = aether_embeddings
# else:
# raise ValueError(f"Unsupported loc_embedding_type: {loc_embedding_type}")

df_stay_filtered


In [ ]:
# Map string user_ids to integer user_id_num
user_id_to_num = {uid: idx for idx, uid in enumerate(df_stay_filtered['user_id'].unique())}
df_stay_filtered['user_id_num'] = df_stay_filtered['user_id'].map(user_id_to_num)

# Optionally, create a reverse mapping if needed
user_id_num_to_str = {v: k for k, v in user_id_to_num.items()}

# Show mapping and updated DataFrame
print('user_id_to_num:', list(user_id_to_num.items())[:5])
df_stay_filtered


In [ ]:
# convert user_id_num_to_str to df and save as csv
df_user_id_mapping = pd.DataFrame(list(user_id_num_to_str.items()), columns=['user_id_num', 'user_id'])
df_user_id_mapping.to_csv(f'{save_dir}/user_id_mapping.csv', index=False)  


In [ ]:
print(user_id_num_to_str[0])
print(len(user_id_num_to_str))

# 000088cc-b646-409b-a026-69cd9d47b838
# 113529


In [ ]:
# check duplicated rows -- user_id and start_time
duplicated_rows = df_stay_filtered[df_stay_filtered.duplicated(subset=['user_id', 'start_time'], keep=False)]
print(f"Number of duplicated rows based on user_id and start_time: {len(duplicated_rows)}")
print(f"Number of unique user_id: {df_stay_filtered['user_id'].nunique()}")


In [ ]:
df_stay_filtered


In [ ]:
# # Get start_min, dur_min
df_stay_filtered['start_time'] = pd.to_datetime(df_stay_filtered['start_time'])
df_stay_filtered['end_time'] = pd.to_datetime(df_stay_filtered['end_time'])
df_stay_filtered['duration_s'] = (df_stay_filtered['end_time'] - df_stay_filtered['start_time']).dt.total_seconds()
df_stay_filtered['duration_s'] = df_stay_filtered['duration_s'].astype(int)
df_stay_filtered['duration_m'] = df_stay_filtered['duration_s'] // 60

df_stay_filtered['hour'] = df_stay_filtered['start_time'].dt.hour
df_stay_filtered['minute'] = df_stay_filtered['start_time'].dt.minute

# Get the day of week (0=Monday, 6=Sunday)
df_stay_filtered['dow'] = df_stay_filtered['start_time'].dt.dayofweek

# df_stay_filtered['end_time'] = pd.to_datetime(df_stay_filtered['end_time'])
# df_stay_filtered['start_min'] = df_stay_filtered['start_time'].dt.hour * 60 + df_stay_filtered['start_time'].dt.minute
# df_stay_filtered['dur_min'] = (df_stay_filtered['end_time'] - df_stay_filtered['start_time']).dt.total_seconds() / 60
df_stay_filtered


In [ ]:
df_stay_filtered.describe()


# Data Preparation and Dataloader


In [ ]:
# Reuse the production data and batching helpers instead of redefining them in the notebook.
from data_preparation import prepare_activity_chain_index
from activity_chain_dataset import (
    ActivityChainDataset,
    HybridPKSampler,
    ace_collate_fn,
    build_split_from_users,
    build_weekpart_identity_map,
)


In [ ]:
# 1. Prepare Index
act_chains, user_map = prepare_activity_chain_index(df_stay_filtered)

count = 0
for k in user_map.keys():
    print(k, user_map[k])
    count += 1
    if count >= 10:
        break


## Sanity check


In [ ]:
# 2. Create Sampler
weekpart_identity_map = build_weekpart_identity_map(act_chains)
hybrid_sampler = HybridPKSampler(weekpart_identity_map, batch_size=256, single_user_ratio=0.5)

print(f"The number of multi-day user/weekpart identities: {len(hybrid_sampler.multi_day_users)}")
print(f"The number of single-day user/weekpart identities: {len(hybrid_sampler.single_day_users)}")
print(f"The number of batches per epoch: {len(hybrid_sampler)}")  

# Previous run we have:
# The number of multi-day users: 35982
# The number of single-day users: 8050
# The number of batches per epoch: 2248

# Most recent run we have: this is without accounting for weekpart (i.e., weekday/weekend) -- we only consider user_id for sampling, not (user_id, weekpart) identity
# The number of multi-day users: 95303
# The number of single-day users: 18226
# The number of batches per epoch: 1489


In [ ]:
act_chains[0]


In [ ]:
# 3. Create Dataset
dataset = ActivityChainDataset(act_chains, loc_embedding_type)

# 4. Create DataLoader
# Note: batch_sampler is mutually exclusive with batch_size/shuffle/sampler
mam_mask_prob = 0.20  # the value is 0.15 in BERT. We use a slightly higher value to encourage more aggressive masking and better generalization. You can tune this hyperparameter.
min_mam_masks_per_chain = 0

def ace_collate_with_masking(batch):
    return ace_collate_fn(
        batch,
        mam_mask_prob=mam_mask_prob,
        min_mam_masks_per_sequence=min_mam_masks_per_chain
    )

dataloader = DataLoader(dataset, batch_sampler=hybrid_sampler, collate_fn=ace_collate_with_masking)


In [ ]:
one_batch = next(iter(dataloader))
one_batch


In [ ]:
# Check the norm of the location embeddings to see if they are normalized
tmp_inspect = one_batch['loc_embeddings'][1][1]
embed_norm = torch.linalg.vector_norm(tmp_inspect)
print(f"Norm of the location embedding vector: {embed_norm.item():.4f}")


In [ ]:
one_batch['loc_embeddings'].shape


In [ ]:
one_batch['dow'].shape


# Models


In [ ]:
# Reuse the production ACE model definitions.
from models import ActivityChainEncoder, PositionalEncoding, TemporalEmbedding


In [ ]:
# Reuse the production loss definitions.
from models import ContrastiveMAMLoss, SupConLoss


In [ ]:
# Sanity check: minimal training step for ActivityChainEncoder with SupConLoss and ContrastiveMAMLoss

# 1. Prepare config and instantiate model
base_config = {
    'd_input': 128,
    'nhead': 8,
    'num_encoder_layers': 4,
    'dim_feedforward': 256,
    'device': 'cuda:0',
    'mam_mask_mode': 'location_only',
    'user_embedding_mode': 'prompt',
    'use_prompt_token': True,
}

# Keep `config` and `model` defined for the later training cells.
config = base_config.copy()

device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')


In [ ]:
# 2. Get one batch from dataloader
batch = next(iter(dataloader))
loc_embeddings = batch['loc_embeddings'].to(device)
start_hour = batch['start_hour_indices'].to(device)
start_minute = batch['start_minute_indices'].to(device)
duration = batch['dur_indices'].to(device)
dow = batch['dow'].to(device)
padding_mask = batch['padding_mask'].to(device)
mam_mask = batch['mam_mask'].to(device)
labels = batch['labels'].to(device)

# 3. Check the configurable encoder variants
variant_configs = [
    {'mam_mask_mode': 'location_only', 'user_embedding_mode': 'prompt', 'use_prompt_token': True},
    {'mam_mask_mode': 'location_only', 'user_embedding_mode': 'mean', 'use_prompt_token': False},
    {'mam_mask_mode': 'location_only', 'user_embedding_mode': 'prompt_mean_concat', 'use_prompt_token': True},
]

supcon_loss_fn = SupConLoss()
mam_loss_fn = ContrastiveMAMLoss()
supcon_weight = 1.0
mam_weight = 0.5

for variant in variant_configs:
    variant_config = base_config.copy()
    variant_config.update(variant)
    test_model = ActivityChainEncoder(variant_config).to(device)
    test_model.train()

    user_embedding, reconstruction_logits = test_model(
        loc_embeddings, start_hour, start_minute, duration, dow, padding_mask, mam_mask
    )
    assert user_embedding.shape == (loc_embeddings.size(0), variant_config['d_input'])
    assert reconstruction_logits.shape == loc_embeddings.shape

    supcon_loss = supcon_loss_fn(user_embedding, labels)
    mam_loss = mam_loss_fn(reconstruction_logits, loc_embeddings, mam_mask)
    total_loss = supcon_weight * supcon_loss + mam_weight * mam_loss
    total_loss.backward()

    print(
        f"Variant {variant}: user_embedding={tuple(user_embedding.shape)}, "
        f"reconstruction={tuple(reconstruction_logits.shape)}, "
        f"SupCon={supcon_loss.item():.4f}, MAM={mam_loss.item():.4f}, "
        f"Weighted Total={total_loss.item():.4f}"
    )

zero_mask_loss = mam_loss_fn(reconstruction_logits, loc_embeddings, torch.zeros_like(mam_mask))
assert zero_mask_loss.item() == 0.0

# Re-create the default model for the later full training cells.
model = ActivityChainEncoder(config).to(device)
print("All configurable encoder sanity checks passed.")


In [ ]:
reconstruction_logits.shape


# Training


In [ ]:
from ace_experiment import make_ace_experiment_paths as _make_ace_experiment_paths


def make_ace_experiment_paths(save_dir, city, loc_embedding_type, experiment_name, create_dirs=True, update_globals=True):
    """Notebook-compatible wrapper around the production path helper."""
    paths = _make_ace_experiment_paths(
        save_dir,
        city,
        loc_embedding_type,
        experiment_name,
        create_dirs=create_dirs,
    )
    if update_globals:
        globals().update(paths)
    return paths


# 1. Prepare config and instantiate model
base_config = {
    'd_input': 128,
    'nhead': 8,
    'num_encoder_layers': 4,
    'dim_feedforward': 256,
    'device': 'cuda:0',
    'mam_mask_mode': 'location_only',  # {'all', 'location_only'}
    'user_embedding_mode': 'prompt',  # {'prompt', 'mean', 'prompt_mean_concat'}
    'use_prompt_token': True,
}

# Keep `config` and `model` defined for the later training cells.
config = base_config.copy()

device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')


In [ ]:
from ace_experiment import plot_training_history
from training import (
    eval_one_epoch as _eval_one_epoch,
    run_training_with_early_stopping as _run_training_with_early_stopping,
    train_one_epoch as _train_one_epoch,
)


def train_one_epoch(
    model, dataloader, supcon_loss_fn, mam_loss_fn, optimizer, device,
    supcon_weight=1.0, mam_weight=1.0
):
    return _train_one_epoch(
        model,
        dataloader,
        supcon_loss_fn,
        mam_loss_fn,
        optimizer,
        device,
        supcon_weight,
        mam_weight,
        show_progress_bars=True,
    )


def eval_one_epoch(
    model, dataloader, supcon_loss_fn, mam_loss_fn, device,
    supcon_weight=1.0, mam_weight=1.0
):
    metrics = _eval_one_epoch(
        model,
        dataloader,
        supcon_loss_fn,
        mam_loss_fn,
        device,
        supcon_weight,
        mam_weight,
    )
    return metrics, torch.empty(0), torch.empty(0, dtype=torch.long)


def run_training_with_early_stopping(
    model, train_loader, val_loader, supcon_loss_fn, mam_loss_fn, optimizer, device,
    num_epochs=50, patience=5, save_path='best_model.pt', plot_losses=True,
    plot_save_path=None, supcon_weight=1.0, mam_weight=1.0
):
    run_config = {
        'num_epochs': num_epochs,
        'patience': patience,
        'supcon_weight': supcon_weight,
        'mam_weight': mam_weight,
        'show_progress_bars': True,
    }
    paths = {'save_path': save_path}
    model, best_val_loss, history = _run_training_with_early_stopping(
        model,
        train_loader,
        val_loader,
        optimizer,
        device,
        paths,
        run_config,
    )
    if plot_losses and plot_save_path is not None:
        plot_training_history(history, plot_save_path)
    return model, best_val_loss, history


# ================================
# Trajectory-User Linking Evaluation
# ================================
def linking_accuracy(query_emb, query_labels, gallery_emb, gallery_labels):
    query_emb = F.normalize(query_emb, dim=1)
    gallery_emb = F.normalize(gallery_emb, dim=1)
    sim = torch.matmul(query_emb, gallery_emb.T)
    pred_idx = sim.argmax(dim=1)
    pred_labels = gallery_labels[pred_idx]
    return (pred_labels == query_labels).float().mean().item()


In [ ]:
import json

# ================================
# 5. Train/Val Split + Example Usage
# ================================

def build_split_from_users(activity_chains, user_to_indices, selected_users):
    split_chains = []
    split_user_map = defaultdict(list)

    for uid in selected_users:
        for old_idx in user_to_indices[uid]:
            new_idx = len(split_chains)
            split_chains.append(activity_chains[old_idx])
            split_user_map[uid].append(new_idx)

    return split_chains, split_user_map


# User-level split to reduce identity leakage between train and val
val_ratio = 0.2
split_seed = 101
batch_size = 256
single_user_ratio = 0.5
mam_mask_prob = 0.20
min_mam_masks_per_chain = 0
supcon_weight = 1.0
mam_weight = 1.0
# city = 'london'  # city is defined at the beginning

# Create a self-contained experiment directory for this ACE run.
experiment_timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
experiment_name = (
    f"{experiment_timestamp}_"
    f"mam-{config.get('mam_mask_mode', 'location_only')}_"
    f"emb-{config.get('user_embedding_mode', 'prompt')}_"
    f"prompt-{int(config.get('use_prompt_token', True))}_"
    f"mamp{mam_mask_prob:g}_minmask{min_mam_masks_per_chain}_"
    f"sw{supcon_weight:g}_mamw{mam_weight:g}"
)
experiment_paths = make_ace_experiment_paths(save_dir, city, loc_embedding_type, experiment_name)
print(f"Experiment directory: {experiment_paths['experiment_dir']}")


In [ ]:
all_users = list(user_map.keys())
if len(all_users) < 2:
    raise ValueError('Need at least 2 users to create both train and validation sets.')

rng = random.Random(split_seed)
rng.shuffle(all_users)

n_val_users = int(len(all_users) * val_ratio)
n_val_users = max(1, min(n_val_users, len(all_users) - 1))

val_users = set(all_users[:n_val_users])
train_users = set(all_users[n_val_users:])

train_chains, train_user_map = build_split_from_users(act_chains, user_map, train_users)
val_chains, val_user_map = build_split_from_users(act_chains, user_map, val_users)

train_dataset = ActivityChainDataset(train_chains, loc_embedding_type)
val_dataset = ActivityChainDataset(val_chains, loc_embedding_type)

train_identity_map = build_weekpart_identity_map(train_chains)
val_identity_map = build_weekpart_identity_map(val_chains)

train_sampler = HybridPKSampler(train_identity_map, batch_size=batch_size, single_user_ratio=single_user_ratio)
val_sampler = HybridPKSampler(val_identity_map, batch_size=batch_size, single_user_ratio=single_user_ratio)

def ace_collate_with_masking(batch):
    return ace_collate_fn(
        batch,
        mam_mask_prob=mam_mask_prob,
        min_mam_masks_per_sequence=min_mam_masks_per_chain
    )

train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=ace_collate_with_masking)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, collate_fn=ace_collate_with_masking)

print(f'Train users: {len(train_users)}, Val users: {len(val_users)}')
print(f'Train user/weekpart identities: {len(train_identity_map)}, Val user/weekpart identities: {len(val_identity_map)}')
print(f'Train chains: {len(train_dataset)}, Val chains: {len(val_dataset)}')
print(f'Train batches/epoch: {len(train_loader)}, Val batches/epoch: {len(val_loader)}')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model, best_val_loss, history = run_training_with_early_stopping(
    model, train_loader, val_loader, SupConLoss(), ContrastiveMAMLoss(), optimizer, device,
    num_epochs=50, patience=5, save_path=save_path, plot_losses=True,
    plot_save_path=training_loss_plot_path,
    supcon_weight=supcon_weight, mam_weight=mam_weight
)

# Save experiment settings and training curves next to the checkpoint.
experiment_config = {
    'city': city,
    'loc_embedding_type': loc_embedding_type,
    'save_dir': save_dir,
    'experiment_name': experiment_name,
    'experiment_dir': experiment_dir,
    'model_config': config,
    'val_ratio': val_ratio,
    'split_seed': split_seed,
    'batch_size': batch_size,
    'single_user_ratio': single_user_ratio,
    'mam_mask_prob': mam_mask_prob,
    'min_mam_masks_per_chain': min_mam_masks_per_chain,
    'supcon_weight': supcon_weight,
    'mam_weight': mam_weight,
    'best_val_loss': best_val_loss,
    'best_model_path': save_path,
    'training_loss_plot_path': training_loss_plot_path,
    'train_users': len(train_users),
    'val_users': len(val_users),
    'train_chains': len(train_dataset),
    'val_chains': len(val_dataset),
}

pd.DataFrame(history).to_csv(training_history_path, index_label='epoch')

with open(experiment_config_path, 'w') as f:
    json.dump(experiment_config, f, indent=2)

print(f'Training history saved to {training_history_path}')
print(f'Experiment config saved to {experiment_config_path}')


In [ ]:
# The total training time is 30 min 41.7 seconds


## Generate Activity Profile Embeddings


In [ ]:
# ================================
# Load Pre-trained ACE Checkpoint
# ================================

load_ace_checkpoint = True

# You need to manually specify the experiment name to load if you want to load from a previous run.
ace_experiment_name_to_load = '20260605_143748_mam-location_only_emb-prompt_prompt-1_mamp0.2_minmask0_sw1_mamw1'

# If you are resuming from an existing experiment without rerunning training,
# define `experiment_name` above this cell. The directory variables below will be rebuilt.
ace_experiment_name_to_load = globals().get('ace_experiment_name_to_load', globals().get('experiment_name', None))
if ace_experiment_name_to_load is not None:
    experiment_name = ace_experiment_name_to_load
    experiment_paths = make_ace_experiment_paths(save_dir, city, loc_embedding_type, experiment_name)

if load_ace_checkpoint:
    ace_checkpoint_path = globals().get('save_path')
    if ace_checkpoint_path is None:
        raise ValueError(
            "No ACE checkpoint path is defined. Run the training setup cell first, "
            "or set `experiment_name` before this cell so `save_path` can be rebuilt."
        )
    if not os.path.exists(ace_checkpoint_path):
        raise FileNotFoundError(
            f"ACE checkpoint not found at {ace_checkpoint_path}. "
            "Train the model first or set `experiment_name` to an existing experiment."
        )

    # Prefer the saved experiment config when available so the loaded architecture matches the checkpoint.
    if 'experiment_config_path' in globals() and os.path.exists(experiment_config_path):
        with open(experiment_config_path, 'r') as f:
            loaded_experiment_config = json.load(f)
        config = loaded_experiment_config.get('model_config', config)
        print(f"Loaded ACE model config from {experiment_config_path}")

    model = ActivityChainEncoder(config).to(device)
    ace_state_dict = torch.load(ace_checkpoint_path, map_location=device)
    model.load_state_dict(ace_state_dict)
    model.eval()
    print(f"Loaded pre-trained ACE checkpoint from {ace_checkpoint_path}")


In [ ]:
# ================================
# Generate Activity Profile Embeddings
# ================================
from training import generate_user_embeddings

user_embeddings, all_embeddings = generate_user_embeddings(
    model,
    act_chains,
    user_map,
    device,
    loc_embedding_type,
)

sample_user_record = next(iter(user_embeddings.values()))
print(f'Generated embeddings for {len(user_embeddings)} users')
print(f"Overall embedding dimension: {sample_user_record['overall_embedding'].shape[0]}")
print(f"Users with weekday embeddings: {sum(v['weekday_embedding'] is not None for v in user_embeddings.values())}")
print(f"Users with weekend embeddings: {sum(v['weekend_embedding'] is not None for v in user_embeddings.values())}")
print(f"Sample user embedding keys: {list(sample_user_record.keys())}")


In [ ]:
# Save embeddings to local path

# Save user embeddings
os.makedirs(embedding_dir, exist_ok=True)
torch.save(user_embeddings, embedding_save_path)
print(f'User embeddings saved to {embedding_save_path}')

# Save per-chain embeddings for optional diagnostics
torch.save(all_embeddings, chain_embedding_save_path)
print(f'Chain embeddings saved to {chain_embedding_save_path}')

# Save metadata
sample_user_record = next(iter(user_embeddings.values()))
metadata = {
    'num_users': len(user_embeddings),
    'embedding_dim': sample_user_record['overall_embedding'].shape[0],
    'embedding_fields': ['overall_embedding', 'weekday_embedding', 'weekend_embedding'],
    'num_users_with_weekday_embedding': sum(v['weekday_embedding'] is not None for v in user_embeddings.values()),
    'num_users_with_weekend_embedding': sum(v['weekend_embedding'] is not None for v in user_embeddings.values()),
    'user_ids': list(user_embeddings.keys()),
    'experiment_name': experiment_name,
    'experiment_dir': experiment_dir,
    'model_config': config,
    'best_model_path': save_path,
}
torch.save(metadata, metadata_save_path)
print(f'Metadata saved to {metadata_save_path}')

print(f'Total user embedding records saved: {len(user_embeddings)}')
